# Voxtral 4B TTS — Apple Silicon (MLX) Guide

This notebook uses **`mlx-community/Voxtral-4B-TTS-2603-mlx-4bit`** — the 4-bit quantised MLX version of Mistral's frontier open-weights TTS model, optimised for Apple Silicon (M1/M2/M3/M4).

| Variant | RAM Usage | RTF (Apple Silicon) | Notes |
|---|---|---|---|
| `mlx-4bit` | ~2.5 GB | 0.97x | ✅ Recommended for 16 GB M4 |
| `mlx-6bit` | ~3.5 GB | 1.15x | Better quality |
| `mlx-bf16` | ~8 GB | 6.50x | Full precision |

> **RTF < 1.0 = faster than real-time.** The 4-bit model runs near real-time at ~2.5 GB — leaves plenty of headroom on a 16 GB M4 Mac.


## 1. Prerequisites

### System Requirements
- macOS 13.3+ (Ventura or later)
- Apple Silicon Mac (M1/M2/M3/M4) — this notebook will **not** work via MLX on Intel Macs
- Python 3.10+
- ~2.5 GB free disk space for the 4-bit model cache

### Model to Download
The model is auto-downloaded on first use by `mlx-audio` to `~/.cache/huggingface/hub`.

| Model ID | Size | Best for |
|---|---|---|
| `mlx-community/Voxtral-4B-TTS-2603-mlx-4bit` | ~2.5 GB | ✅ 16 GB RAM |
| `mlx-community/Voxtral-4B-TTS-2603-mlx-6bit` | ~3.5 GB | Better quality |
| `mlx-community/Voxtral-4B-TTS-2603-mlx-bf16` | ~8 GB | Full precision |

### One-time setup (do this before running cells)
1. Accept the model license at: https://huggingface.co/mistralai/Voxtral-4B-TTS-2603
2. Create a Hugging Face access token at: https://huggingface.co/settings/tokens
3. Either run `huggingface-cli login` in terminal, or set `HF_TOKEN` in the cell below

In [1]:
import os

from dotenv import load_dotenv
load_dotenv()
hf_api_key = os.getenv("hf_api_key")

from huggingface_hub import login, whoami
login(token=hf_api_key, add_to_git_credential=False)

info = whoami()
print(f"✓ Logged in as: {info['name']}")

# Verify MLX is available (Apple Silicon only)
try:
    import mlx.core as mx
    print(f"✅ MLX available")
    print(f"   Default device : {mx.default_device()}")
    print(f"   Metal GPU      : {mx.metal.is_available()}")
except ImportError:
    print("❌ MLX not found — ensure you're on Apple Silicon and mlx-audio is installed.")

✓ Logged in as: *********
✅ MLX available
   Default device : Device(gpu, 0)
   Metal GPU      : True


## 2. Load the Model

`mlx-audio` automatically downloads the model to `~/.cache/huggingface/hub` on first run (~2.5 GB).  
Subsequent runs load directly from cache — no internet required.

In [2]:
import sys

# Show which Python the kernel is using
print(f"Kernel Python: {sys.executable}")

# Install mlx-audio with the tts extra (includes tiktoken) + soundfile
!"{sys.executable}" -m pip install -q -U "mlx-audio[tts]" soundfile tiktoken

# Verify the install is visible
import importlib.util
for pkg in ["mlx_audio", "soundfile", "tiktoken"]:
    found = importlib.util.find_spec(pkg) is not None
    print(f"  {'✅' if found else '❌'} {pkg} {'found' if found else 'NOT found'} in kernel path")

Kernel Python: /Tutorials/prompt-shot/.venv/bin/python3
/Tutorials/prompt-shot/.venv/bin/python3: No module named pip
  ✅ mlx_audio found in kernel path
  ✅ soundfile found in kernel path
  ✅ tiktoken found in kernel path


In [3]:
from mlx_audio.tts.utils import load

# Change to mlx-6bit or mlx-bf16 if you prefer higher quality and have the RAM
MODEL_ID = "mlx-community/Voxtral-4B-TTS-2603-mlx-4bit"

print(f"Loading: {MODEL_ID}")
print("First run downloads ~2.5 GB — subsequent runs use cache.\n")

model = load(MODEL_ID)
print("✅ Model loaded successfully.")

Loading: mlx-community/Voxtral-4B-TTS-2603-mlx-4bit
First run downloads ~2.5 GB — subsequent runs use cache.



Fetching 25 files:   0%|          | 0/25 [00:00<?, ?it/s]

Loaded local Tekken tokenizer from /.cache/huggingface/hub/models--mlx-community--Voxtral-4B-TTS-2603-mlx-4bit/snapshots/f98fc91b9cb5adc7dab56102c690458276c14c6a/tekken.json
✅ Model loaded successfully.


## 3. Available Voices

Voxtral 4B TTS includes **20 preset voices** across 9 languages:

| Language | Voice IDs |
|---|---|
| 🇬🇧 English | `casual_male`, `casual_female`, `cheerful_female`, `neutral_male`, `neutral_female` |
| 🇫🇷 French | `fr_male`, `fr_female` |
| 🇪🇸 Spanish | `es_male`, `es_female` |
| 🇩🇪 German | `de_male`, `de_female` |
| 🇮🇹 Italian | `it_male`, `it_female` |
| 🇵🇹 Portuguese | `pt_male`, `pt_female` |
| 🇳🇱 Dutch | `nl_male`, `nl_female` |
| 🇸🇦 Arabic | `ar_male` |
| 🇮🇳 Hindi | `hi_male`, `hi_female` |

## 4. Generate Audio

The model streams audio chunks as they are generated. We collect all chunks then save one WAV file at 24 kHz.

In [4]:
import soundfile as sf
import numpy as np
import os

# --- Configuration ---
TEXT        = "In today's MysteryBytes Labs session, we are learning to generate studio quality audio using \
               Voxtral TTS on our local machine with zero subscription cost. Share your favourite voice in comments. "
VOICE       = "casual_male"      # See voice table above
OUTPUT_FILE = "outputs/voxtral_output.wav"
SAMPLE_RATE = 24_000             # Voxtral outputs 24 kHz audio

os.makedirs("outputs", exist_ok=True)

def _duration_secs(d) -> float:
    """Convert audio_duration to seconds — handles both float and 'HH:MM:SS.mmm' strings."""
    if isinstance(d, str):
        parts = d.split(":")
        return sum(float(p) * 60 ** (len(parts) - 1 - i) for i, p in enumerate(parts))
    return float(d)

print(f"Voice : {VOICE}")
print(f"Text  : {TEXT}\n")

audio_chunks = []

for result in model.generate(text=TEXT, voice=VOICE):
    chunk = np.array(result.audio, dtype=np.float32)
    audio_chunks.append(chunk)
    print(f"  Received chunk — {_duration_secs(result.audio_duration):.2f}s", end="\r")

audio_data = np.concatenate(audio_chunks)
sf.write(OUTPUT_FILE, audio_data, SAMPLE_RATE)

total_duration = len(audio_data) / SAMPLE_RATE
print(f"\n✅ Saved '{OUTPUT_FILE}'  ({total_duration:.2f}s total, {len(audio_data)} samples)")

Voice : casual_male
Text  : In today's MysteryBytes Labs session, we are learning to generate studio quality audio using                Voxtral TTS on our local machine with zero subscription cost. Share your favourite voice in comments. 

  Received chunk — 10.88s
✅ Saved 'outputs/voxtral_output.wav'  (10.88s total, 261120 samples)


## 5. Play the Audio

Listen to the generated audio directly in the notebook.

In [5]:
from IPython.display import Audio, display

display(Audio(OUTPUT_FILE))

## 6. Compare Multiple Voices

Generate the same sentence with several English voices side-by-side.

In [6]:
DEMO_TEXT   = "The quick brown fox jumps over the lazy dog near the riverbank."
DEMO_VOICES = ["casual_male", "casual_female", "cheerful_female", "neutral_male", "neutral_female"]

for voice in DEMO_VOICES:
    chunks = []
    for result in model.generate(text=DEMO_TEXT, voice=voice):
        chunks.append(np.array(result.audio, dtype=np.float32))
    audio = np.concatenate(chunks)
    path = f"outputs/voxtral_{voice}.wav"
    sf.write(path, audio, SAMPLE_RATE)
    print(f"🔊 {voice}")
    display(Audio(path))
    print()

🔊 casual_male



🔊 casual_female



🔊 cheerful_female



🔊 neutral_male



🔊 neutral_female


## 7. Multilingual Examples

Voxtral natively supports 9 languages with zero switching overhead.

In [7]:
MULTILINGUAL = [
    ("fr", "fr_male","Bonjour, je suis un assistant vocal multilingue."),
]

for lang, voice, text in MULTILINGUAL:
    chunks = []
    for result in model.generate(text=text, voice=voice):
        chunks.append(np.array(result.audio, dtype=np.float32))
    audio = np.concatenate(chunks)
    path = f"outputs/voxtral_{lang}.wav"
    sf.write(path, audio, SAMPLE_RATE)
    print(f"[{lang.upper()}] {voice}")
    print(f"  {text}")
    display(Audio(path))
    print()

[FR] fr_male
  Bonjour, je suis un assistant vocal multilingue.
